# Qwen3 Base Interactive Fairy-tale Engine v9

원본 Qwen만 사용하는 테스트 노트북입니다.  
주제는 더 단순한 고전형 플롯으로 바꿨습니다: **용에게 납치된 공주를 구하러 가는 모험**.

핵심 변경:
- LoRA 사용 안 함
- 특정 단어 블랙리스트 없음
- 선택지 템플릿 fallback 없음
- 선택지는 LLM 생성 + 재시도만 사용
- 이전 장면 전체를 다음 프롬프트에 넣지 않고, 요약/마지막 문장/상태만 전달
- 선택한 선택지는 `choice_start_sentence`로 다음 장면 첫 문장을 강제
- 이전 장면 복사 overlap 검사
- decision_context에 실제 등장한 anchor만 선택지에 사용
- stage pacing 적용


In [ ]:
# ============================================================
# 1. Install
# ============================================================
!pip -q install -U "transformers>=4.51.0" accelerate bitsandbytes sentencepiece huggingface_hub


In [ ]:
# ============================================================
# 2. HF login / cache
# ============================================================
from huggingface_hub import login
import os

# 필요하면 주석 해제
# login()

HF_CACHE = "/content/hf_cache"
os.makedirs(HF_CACHE, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE
os.environ["HF_HUB_CACHE"] = HF_CACHE
print("HF_CACHE:", HF_CACHE)


In [ ]:
# ============================================================
# 3. Settings
# ============================================================
import torch

# T4 안정 기본값. 14B는 T4에서 느리고 불안정할 수 있음.
MODEL_ID = "Qwen/Qwen3-8B"
# MODEL_ID = "Qwen/Qwen3-14B"

USE_4BIT = True
MAX_INPUT_TOKENS = 3072
STORY_MAX_NEW_TOKENS = 1100
JSON_MAX_NEW_TOKENS = 750

print("MODEL_ID:", MODEL_ID)


In [ ]:
# ============================================================
# 4. Load original Qwen base model only
# ============================================================
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = None
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    cache_dir=HF_CACHE,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    cache_dir=HF_CACHE,
)

model.eval()
print("Loaded:", MODEL_ID)


In [ ]:
# ============================================================
# 5. Generation utils
# ============================================================
import json, re, math, random
from pprint import pprint


def model_device():
    try:
        return next(model.parameters()).device
    except Exception:
        return "cuda" if torch.cuda.is_available() else "cpu"


def strip_artifacts(text):
    text = str(text)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = text.replace("<|im_end|>", "").replace("<|endoftext|>", "")
    text = text.replace("```json", "").replace("```", "")
    return text.strip()


def build_chat_prompt(messages):
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )


@torch.inference_mode()
def generate_raw(messages, max_new_tokens=900, temperature=0.5, top_p=0.84,
                 repetition_penalty=1.12, max_input_tokens=None):
    if max_input_tokens is None:
        max_input_tokens = MAX_INPUT_TOKENS
    prompt = build_chat_prompt(messages)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_tokens,
    )
    device = model_device()
    inputs = {k: v.to(device) for k, v in inputs.items()}
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        repetition_penalty=repetition_penalty,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    gen = out[0][inputs["input_ids"].shape[-1]:]
    return strip_artifacts(tokenizer.decode(gen, skip_special_tokens=True))


def extract_json_obj(text):
    text = strip_artifacts(text)
    start = text.find("{")
    end = text.rfind("}")
    if start < 0 or end <= start:
        return None, text
    cand = text[start:end+1].strip()
    for c in [cand, re.sub(r",\s*([}\]])", r"\1", cand)]:
        try:
            return json.loads(c), c
        except Exception:
            pass
    return None, cand


def generate_json(messages, max_new_tokens=700, temperature=0.28, top_p=0.76,
                  repetition_penalty=1.05, retry=1, max_input_tokens=None):
    last_raw = ""
    last_cand = ""
    for attempt in range(retry + 1):
        raw = generate_raw(
            messages,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            max_input_tokens=max_input_tokens or MAX_INPUT_TOKENS,
        )
        obj, cand = extract_json_obj(raw)
        last_raw, last_cand = raw, cand
        if obj is not None:
            return obj, raw
        messages = messages + [{
            "role": "user",
            "content": "JSON 파싱에 실패했다. 설명 없이 올바른 JSON 객체만 다시 출력해라."
        }]
    print("[JSON PARSE FAILED]")
    print(last_cand)
    return None, last_raw


In [ ]:
# ============================================================
# 6. Text / entity utils
# ============================================================

def normalize_entity(e):
    if e is None:
        return ""
    e = str(e).strip()
    e = e.strip(" \t\n\r\"'“”‘’『』「」《》[](){}<>")
    e = " ".join(e.split())
    return e.strip()


def clean_list(xs):
    if xs is None:
        return []
    if isinstance(xs, str):
        xs = [xs]
    if not isinstance(xs, list):
        return []
    out = []
    for x in xs:
        x = normalize_entity(x)
        if len(x) >= 2:
            out.append(x)
    return list(dict.fromkeys(out))


def split_sentences_ko(text):
    text = re.sub(r"\s+", " ", str(text).strip())
    if not text:
        return []
    parts = re.split(r"(?<=[.!?])\s+|(?<=[다요까죠])\s+", text)
    return [p.strip() for p in parts if len(p.strip()) >= 2]


def short_name(name):
    name = normalize_entity(name)
    if not name:
        return "주인공"
    return name.split()[-1]


def has_latin_noise(text):
    # 한국어 동화 본문 안에 떨yer 같은 영문 노이즈 탐지
    return bool(re.search(r"[A-Za-z]{2,}", str(text)))


def text_overlap_ratio(a, b, n=8):
    def grams(x):
        x = re.sub(r"\s+", "", str(x))
        if len(x) < n:
            return set()
        return set(x[i:i+n] for i in range(len(x)-n+1))
    ga, gb = grams(a), grams(b)
    if not ga or not gb:
        return 0.0
    return len(ga & gb) / max(1, len(gb))


def entity_match(entity, allowed):
    entity = normalize_entity(entity)
    if not entity:
        return True
    allowed = clean_list(allowed)
    for a in allowed:
        if entity == a or entity in a or a in entity:
            return True
    return False


def anchor_pool(anchors):
    if not isinstance(anchors, dict):
        return []
    pool = []
    for key in ["characters", "objects", "places", "clues", "problems", "visible_items"]:
        pool += clean_list(anchors.get(key, []))
    return list(dict.fromkeys(pool))


def choice_text(choice):
    if isinstance(choice, dict):
        return str(choice.get("text", "")).strip()
    return str(choice).strip()


In [ ]:
# ============================================================
# 7. Classic story config: dragon kidnapped princess
# ============================================================

WORLD_CONFIG = {
    "title_hint": "용의 탑에 갇힌 공주",
    "theme": "용에게 납치된 공주를 구하러 가는 모험",
    "target_age": "초등 저학년",
    "tone": "따뜻하고 긴장감은 있지만 잔인하지 않은 판타지 동화",
    "world_seed": {
        "heroes": ["소년 기사 레오"],
        "princesses": ["아리아 공주"],
        "dragons": ["붉은 용 그라움"],
        "helpers": ["말하는 까마귀 노아", "대장장이 할머니"],
        "objects": ["은빛 방패", "낡은 지도", "용의 비늘 열쇠", "작은 나침반"],
        "places": ["왕성 앞마당", "검은 숲길", "불꽃 다리", "용의 탑", "탑 꼭대기 방"],
        "problems": ["공주가 용의 탑에 갇힘", "용의 탑으로 가는 길이 막힘", "용은 큰 소리에 잠에서 깨어남"]
    },
    "stage_order": ["opening", "exploration", "crisis", "ending"],
    "stage_min_turns": {
        "opening": 1,
        "exploration": 2,
        "crisis": 2,
        "ending": 1
    },
    "stage_rules": {
        "opening": {
            "purpose": "공주가 납치된 사건을 알고 모험을 시작한다.",
            "allowed": "성문, 왕성 앞마당, 첫 단서, 첫 조력자 등장",
            "forbidden": "용과 직접 싸우기, 공주 구출 성공, 최종 탑 도착"
        },
        "exploration": {
            "purpose": "용의 탑으로 가는 길과 필요한 도구를 알아낸다.",
            "allowed": "검은 숲길, 지도, 방패, 까마귀 노아, 대장장이 할머니, 작은 장애물",
            "forbidden": "공주 구출 성공, 용을 완전히 물리치기"
        },
        "crisis": {
            "purpose": "탑 근처에서 큰 위험을 만나고, 선택과 협력이 필요해진다.",
            "allowed": "불꽃 다리, 용의 그림자, 비늘 열쇠, 길을 잃는 위기",
            "forbidden": "너무 빨리 해피엔딩으로 끝내기"
        },
        "ending": {
            "purpose": "공주를 구하고 용과의 문제를 지혜롭게 해결한다.",
            "allowed": "공주 구출, 용의 오해 해소, 왕국 회복",
            "forbidden": "잔인한 전투 묘사"
        }
    }
}


def init_story_state(config=WORLD_CONFIG):
    seed = config["world_seed"]
    protagonist = seed["heroes"][0]
    return {
        "config": config,
        "story_plan": {},
        "current_stage": "opening",
        "stage_turns": {s: 0 for s in config["stage_order"]},
        "known_facts": [],
        "known_entities": [],
        "open_threads": [],
        "locations": {"current": None, "known": []},
        "inventory": [],
        "last_scene_summary": "",
        "last_scene_tail": "",
        "last_story": "",
        "last_anchors": {},
        "last_decision_context": {},
        "last_choices": [],
        "protagonist_short": short_name(protagonist),
    }


def get_allowed_entities(state, blueprint=None):
    allowed = []
    seed = state.get("config", {}).get("world_seed", {})
    for vals in seed.values():
        allowed += clean_list(vals)
    plan = state.get("story_plan", {})
    for k in ["title", "protagonist", "final_goal", "main_conflict"]:
        allowed += clean_list(plan.get(k, []))
    allowed += clean_list(state.get("known_entities", []))
    if blueprint:
        req = blueprint.get("required_elements", {})
        if isinstance(req, dict):
            for k in ["characters", "objects", "places", "clues", "problems"]:
                allowed += clean_list(req.get(k, []))
    return list(dict.fromkeys([normalize_entity(x) for x in allowed if normalize_entity(x)]))


In [ ]:
# ============================================================
# 8. Story plan
# ============================================================
SYSTEM_BASE = """
너는 초등 저학년용 인터랙티브 동화 작가다.
주어진 세계관과 현재 상태를 지키며, 아이가 계속 선택하고 싶어지는 동화를 만든다.
문체는 부드럽고 구체적이어야 한다. 잔인한 전투 묘사는 피한다.
""".strip()


def create_story_plan(state):
    cfg = state["config"]
    prompt = f"""
TASK: CREATE_CLASSIC_STORY_PLAN

세계관 seed:
{json.dumps(cfg['world_seed'], ensure_ascii=False, indent=2)}

주제: {cfg['theme']}
대상: {cfg['target_age']}
톤: {cfg['tone']}

해야 할 일:
1. 흔한 고전 동화 구조로 계획을 만든다.
2. 용에게 납치된 공주를 구하러 가는 명확한 목표를 유지한다.
3. opening/exploration/crisis/ending의 단계 목표를 만든다.
4. 잔인한 싸움보다 용기, 지혜, 대화, 협력으로 해결하는 방향으로 만든다.

출력 JSON:
{{
  "title": "...",
  "protagonist": "...",
  "final_goal": "...",
  "main_conflict": "...",
  "core_entities": ["..."],
  "emotional_arc": ["..."],
  "chapter_goals": [
    {{"stage": "opening", "goal": "..."}},
    {{"stage": "exploration", "goal": "..."}},
    {{"stage": "crisis", "goal": "..."}},
    {{"stage": "ending", "goal": "..."}}
  ]
}}
""".strip()
    obj, raw = generate_json(
        [{"role": "system", "content": SYSTEM_BASE}, {"role": "user", "content": prompt}],
        max_new_tokens=JSON_MAX_NEW_TOKENS,
        temperature=0.25,
        top_p=0.75,
        retry=1,
    )
    if obj:
        state["story_plan"] = obj
        state["known_entities"] = clean_list(obj.get("core_entities", []))
        state["protagonist_short"] = short_name(obj.get("protagonist", WORLD_CONFIG["world_seed"]["heroes"][0]))
    return obj, raw


In [ ]:
# ============================================================
# 9. Stage / blueprint
# ============================================================

def current_stage_rule(state):
    stage = state.get("current_stage", "opening")
    return state["config"]["stage_rules"].get(stage, {})


def compact_previous_context(state):
    return {
        "last_scene_summary": state.get("last_scene_summary", ""),
        "last_scene_tail": state.get("last_scene_tail", ""),
        "known_facts": state.get("known_facts", [])[-6:],
        "open_threads": state.get("open_threads", [])[-4:],
        "inventory": state.get("inventory", []),
        "current_location": state.get("locations", {}).get("current"),
        "current_stage": state.get("current_stage", "opening"),
    }


def create_scene_blueprint(state, selected_choice=None):
    cfg = state["config"]
    stage = state.get("current_stage", "opening")
    rule = current_stage_rule(state)
    selected_part = "선택된 선택지는 아직 없다. 첫 장면 또는 새 장면의 시작이다."
    if selected_choice:
        selected_part = f"""
사용자가 고른 선택지:
{json.dumps(selected_choice, ensure_ascii=False, indent=2)}

중요:
- 이번 장면은 반드시 선택지의 contract.start_sentence로 시작할 수 있게 설계한다.
- 선택한 행동의 결과가 새 정보 1개와 새 문제 1개를 만든다.
"""

    prompt = f"""
TASK: CREATE_SCENE_BLUEPRINT_V9

이야기 계획:
{json.dumps(state['story_plan'], ensure_ascii=False, indent=2)}

현재 단계: {stage}
단계 규칙:
{json.dumps(rule, ensure_ascii=False, indent=2)}

이전 상태 요약:
{json.dumps(compact_previous_context(state), ensure_ascii=False, indent=2)}

세계관 seed:
{json.dumps(cfg['world_seed'], ensure_ascii=False, indent=2)}

{selected_part}

해야 할 일:
1. 이번 장면의 scene_goal을 만든다.
2. 장면은 새 정보 1개와 새 문제 1개를 반드시 만든다.
3. decision_anchors는 장면 중간에 실행할 행동이 아니라, 마지막 선택지 후보로 남겨둘 요소다.
4. opening에서 공주를 바로 구하거나 용과 직접 싸우면 안 된다.
5. exploration에서 결말로 끝내면 안 된다.
6. story 본문은 쓰지 말고 설계도만 만든다.

출력 JSON:
{{
  "stage": "{stage}",
  "scene_goal": "...",
  "emotional_goal": "...",
  "required_elements": {{
    "characters": ["..."],
    "objects": ["..."],
    "places": ["..."],
    "clues": ["..."],
    "problems": ["..."]
  }},
  "new_information": "...",
  "new_problem": "...",
  "beats": [
    {{"name": "setup", "purpose": "장소와 분위기", "must_include": ["..."]}},
    {{"name": "emotion", "purpose": "주인공 감정", "must_include": ["..."]}},
    {{"name": "action", "purpose": "구체 행동", "must_include": ["..."]}},
    {{"name": "reaction", "purpose": "인물 또는 환경 반응", "must_include": ["..."]}},
    {{"name": "discovery", "purpose": "새 단서", "must_include": ["..."]}},
    {{"name": "complication", "purpose": "바로 해결하지 못하는 문제", "must_include": ["..."]}},
    {{"name": "decision_point", "purpose": "다음 선택지가 필요한 열린 상황", "must_include": ["..."]}}
  ],
  "decision_anchors": ["...", "...", "..."]
}}
""".strip()
    obj, raw = generate_json(
        [{"role": "system", "content": SYSTEM_BASE}, {"role": "user", "content": prompt}],
        max_new_tokens=JSON_MAX_NEW_TOKENS,
        temperature=0.28,
        top_p=0.78,
        retry=1,
    )
    return obj, raw


In [ ]:
# ============================================================
# 10. Story generation with start sentence + no copy
# ============================================================

def build_choice_start_sentence(choice, state):
    contract = choice.get("contract", {}) if isinstance(choice, dict) else {}
    s = normalize_entity(contract.get("start_sentence", ""))
    if s:
        if not s.endswith((".", "요.")):
            s += "."
        return s
    # 이 경우는 실패에 가깝지만, 문장만 강제하기 위해 최소 변환
    txt = choice_text(choice).rstrip(". ")
    return txt + "."


def validate_story(story, previous_story="", min_chars=520, min_sentences=7):
    problems = []
    story = str(story).strip()
    if len(story) < min_chars:
        problems.append(f"장면이 너무 짧음: {len(story)}자 < {min_chars}자")
    sc = len(split_sentences_ko(story))
    if sc < min_sentences:
        problems.append(f"문장 수 부족: {sc}문장 < {min_sentences}문장")
    if has_latin_noise(story):
        problems.append("한국어 동화에 영문 노이즈가 섞임")
    if previous_story:
        ov = text_overlap_ratio(previous_story, story)
        if ov > 0.42:
            problems.append(f"직전 장면과 겹침이 큼: overlap={ov:.2f}")
    return len(problems) == 0, problems


def write_story_from_blueprint(state, blueprint, selected_choice=None, retry=2):
    previous_context = compact_previous_context(state)
    previous_story = state.get("last_story", "")
    start_sentence = None
    start_rule = ""
    if selected_choice:
        start_sentence = build_choice_start_sentence(selected_choice, state)
        start_rule = f"""
다음 story는 반드시 아래 문장으로 시작한다.
{start_sentence}
"""

    fail_notes = []
    for attempt in range(retry + 1):
        prompt = f"""
TASK: WRITE_BEAT_BY_BEAT_SCENE_V9

이야기 계획:
{json.dumps(state['story_plan'], ensure_ascii=False, indent=2)}

현재 단계: {state.get('current_stage')}
단계 규칙:
{json.dumps(current_stage_rule(state), ensure_ascii=False, indent=2)}

이전 장면 전체를 반복하지 마라. 참고용 요약만 사용한다:
{json.dumps(previous_context, ensure_ascii=False, indent=2)}

이번 장면 설계도:
{json.dumps(blueprint, ensure_ascii=False, indent=2)}

{start_rule}

이전 실패 이유:
{json.dumps(fail_notes, ensure_ascii=False, indent=2)}

작성 규칙:
1. 동화 본문만 출력한다. JSON, 제목, beat 이름은 출력하지 않는다.
2. 8~12문장, 한국어 550~850자 정도로 쓴다.
3. 각 beat를 순서대로 자연스럽게 반영한다.
4. 새 정보 1개와 새 문제 1개가 분명히 드러나야 한다.
5. decision_anchors는 마지막에 선택 후보로 남길 뿐, 본문 중간에서 모두 실행하지 않는다.
6. 직전 장면 문장을 그대로 반복하지 않는다.
7. "하였다"보다 "했어요"를 사용한다.
8. 잔인한 전투 묘사는 쓰지 않는다.
9. 마지막 1~2문장은 아이가 다음 행동을 고르고 싶게 열린 상황으로 끝낸다.
""".strip()
        story = generate_raw(
            [{"role": "system", "content": SYSTEM_BASE}, {"role": "user", "content": prompt}],
            max_new_tokens=STORY_MAX_NEW_TOKENS,
            temperature=0.50 if attempt == 0 else 0.42,
            top_p=0.84,
            repetition_penalty=1.16 + attempt*0.04,
            max_input_tokens=MAX_INPUT_TOKENS,
        )
        story = strip_artifacts(story)
        obj, _ = extract_json_obj(story)
        if obj and isinstance(obj, dict) and obj.get("story"):
            story = str(obj["story"]).strip()
        ok, probs = validate_story(story, previous_story=previous_story)
        if start_sentence and not story.startswith(start_sentence):
            probs.append("선택지 시작 문장으로 시작하지 않음")
            ok = False
        if ok:
            return story, []
        fail_notes = probs
    return story, fail_notes


In [ ]:
# ============================================================
# 11. Anchors / decision context / affordances / choices
# ============================================================

def extract_scene_anchors(story):
    prompt = f"""
TASK: EXTRACT_SCENE_ANCHORS

아래 story에 실제로 등장한 요소만 추출해라.
story에 없는 요소는 넣지 마라.
선택지 문장을 넣지 마라.

story:
{story}

출력 JSON:
{{
  "scene_anchors": {{
    "characters": ["..."],
    "objects": ["..."],
    "places": ["..."],
    "clues": ["..."],
    "problems": ["..."],
    "visible_items": ["..."]
  }},
  "generated_entities": ["story 안의 핵심 인물/물건/장소"]
}}
""".strip()
    obj, raw = generate_json(
        [{"role": "system", "content": SYSTEM_BASE}, {"role": "user", "content": prompt}],
        max_new_tokens=JSON_MAX_NEW_TOKENS,
        temperature=0.22,
        top_p=0.72,
        retry=1,
    )
    anchors = obj.get("scene_anchors", {}) if obj else {}
    cleaned = {}
    for k, vals in anchors.items():
        keep = []
        for v in clean_list(vals):
            if v in story or any(tok in story for tok in clean_list(re.findall(r"[가-힣A-Za-z0-9]{2,}", v))):
                keep.append(v)
        cleaned[k] = list(dict.fromkeys(keep))
    gen = clean_list(obj.get("generated_entities", [])) if obj else []
    return {"scene_anchors": cleaned, "generated_entities": gen}, raw


def make_decision_context(state, story, anchors, blueprint):
    sents = split_sentences_ko(story)
    tail = " ".join(sents[-3:])
    pool = anchor_pool(anchors)
    # 마지막 문맥 또는 blueprint decision_anchors에 관련 있는 앵커를 우선 사용
    prompt = f"""
TASK: MAKE_DECISION_CONTEXT

story의 마지막 부분:
{tail}

scene_anchors:
{json.dumps(anchors, ensure_ascii=False, indent=2)}

blueprint decision_anchors:
{json.dumps(blueprint.get('decision_anchors', []), ensure_ascii=False, indent=2)}

anchor 후보:
{json.dumps(pool, ensure_ascii=False, indent=2)}

해야 할 일:
1. 다음 선택지를 만들 decision_context를 만든다.
2. available_anchors는 반드시 anchor 후보 중에서만 고른다.
3. story 마지막 상황에서 지금 당장 행동할 수 있는 요소만 고른다.
4. 새로운 인물/물건/장소를 만들지 마라.

출력 JSON:
{{
  "current_situation": "...",
  "tension": "...",
  "decision_question": "...",
  "available_anchors": ["..."]
}}
""".strip()
    obj, raw = generate_json(
        [{"role": "system", "content": SYSTEM_BASE}, {"role": "user", "content": prompt}],
        max_new_tokens=600,
        temperature=0.25,
        top_p=0.74,
        retry=1,
    )
    dc = obj if obj else {}
    allowed = set(pool)
    dc["available_anchors"] = [a for a in clean_list(dc.get("available_anchors", [])) if a in allowed]
    return dc, raw


def make_affordances(state, decision_context, anchors):
    protagonist = state.get("protagonist_short", "레오")
    prompt = f"""
TASK: MAKE_AFFORDANCES_V9

주인공: {protagonist}

decision_context:
{json.dumps(decision_context, ensure_ascii=False, indent=2)}

scene_anchors:
{json.dumps(anchors, ensure_ascii=False, indent=2)}

규칙:
1. affordance는 available_anchors 중 하나를 anchor로 사용한다.
2. 주인공 자신에게 질문하는 행동은 금지한다.
3. 역할은 단서 확인, 인물에게 질문, 장소 이동, 도구 사용, 조심스러운 접근 중에서 고른다.
4. 각 affordance는 다음 장면의 방향을 다르게 만든다.

출력 JSON:
{{
  "affordances": [
    {{"id": "A", "role": "...", "anchor": "...", "action": "...", "risk": "낮음|중간|높음", "story_effect": "..."}}
  ]
}}
""".strip()
    obj, raw = generate_json(
        [{"role": "system", "content": SYSTEM_BASE}, {"role": "user", "content": prompt}],
        max_new_tokens=650,
        temperature=0.30,
        top_p=0.76,
        retry=1,
    )
    aff = obj.get("affordances", []) if obj else []
    valid = set(clean_list(decision_context.get("available_anchors", [])))
    aff = [a for a in aff if normalize_entity(a.get("anchor")) in valid]
    return aff, raw


def normalize_choice_surface(choice, state):
    if not isinstance(choice, dict):
        return choice
    pname = state.get("protagonist_short", "레오")
    text = choice.get("text", "").strip()
    if text and not text.startswith(pname):
        text = f"{pname}가 {text}"
    # 부드러운 종결 보정. 내용 fallback이 아니라 표면 정규화만 수행.
    if text and not text.endswith(("요.", "봐요.")):
        text = re.sub(r"(한다|하자|간다|찾는다|묻는다)\.?$", "해 봐요.", text)
        if not text.endswith(("요.", "봐요.")):
            text = text.rstrip(" .") + "해 봐요."
    choice["text"] = text
    contract = choice.get("contract", {}) if isinstance(choice.get("contract", {}), dict) else {}
    if contract.get("start_sentence"):
        s = contract["start_sentence"].strip()
        if not s.endswith((".", "요.")):
            s += "."
        contract["start_sentence"] = s
    choice["contract"] = contract
    return choice


def validate_choices(choices, state, decision_context):
    problems = []
    if not isinstance(choices, list) or len(choices) != 3:
        return False, [f"choices 개수 오류: {len(choices) if isinstance(choices, list) else 'not list'}"]
    available = set(clean_list(decision_context.get("available_anchors", [])))
    pname = state.get("protagonist_short", "레오")
    seen = set()
    for i, c in enumerate(choices, 1):
        text = choice_text(c)
        anchor = normalize_entity(c.get("anchor", ""))
        role = c.get("role", "")
        if text in seen:
            problems.append(f"{i}번 선택지 중복 | {text}")
        seen.add(text)
        if not text.startswith(pname):
            problems.append(f"{i}번 선택지가 주인공 이름으로 시작하지 않음 | {text}")
        if anchor not in available:
            problems.append(f"{i}번 anchor가 decision_context에 없음: {anchor} | {text}")
        if anchor and anchor not in text:
            problems.append(f"{i}번 선택지에 anchor가 없음: {anchor} | {text}")
        if "질문" in role and (anchor == pname or pname in anchor):
            problems.append(f"{i}번 주인공 자신에게 질문 | {text}")
        contract = c.get("contract", {}) if isinstance(c.get("contract", {}), dict) else {}
        if not contract.get("start_sentence"):
            problems.append(f"{i}번 contract.start_sentence 없음 | {text}")
    return len(problems) == 0, problems


def make_choices_with_contracts(state, decision_context, affordances):
    pname = state.get("protagonist_short", "레오")
    prompt = f"""
TASK: MAKE_CHOICES_WITH_CONTRACTS_V9

주인공 이름: {pname}

decision_context:
{json.dumps(decision_context, ensure_ascii=False, indent=2)}

affordances:
{json.dumps(affordances, ensure_ascii=False, indent=2)}

규칙:
1. choices는 정확히 3개.
2. 각 choice.anchor는 반드시 decision_context.available_anchors 중 하나다.
3. 각 선택지는 반드시 "{pname}가"로 시작한다.
4. 선택지는 초등 저학년 앱 버튼처럼 "~해 봐요." 말투로 쓴다.
5. 주인공 자신에게 질문하는 선택지는 금지한다.
6. 각 선택지는 서로 다른 결과를 만든다.
7. 각 선택지에는 hidden contract를 붙인다.
8. contract.start_sentence는 다음 장면의 첫 문장으로 바로 쓸 수 있는 자연스러운 과거형 문장이어야 한다.

출력 JSON:
{{
  "choices": [
    {{
      "id": "A",
      "role": "...",
      "anchor": "...",
      "risk": "낮음|중간|높음",
      "text": "...",
      "contract": {{
        "start_sentence": "...",
        "must_include_result": "...",
        "must_reveal": "...",
        "must_create_next_problem": "..."
      }}
    }}
  ]
}}
""".strip()
    messages = [{"role": "system", "content": SYSTEM_BASE}, {"role": "user", "content": prompt}]
    last_probs = []
    for attempt in range(3):
        obj, raw = generate_json(
            messages,
            max_new_tokens=JSON_MAX_NEW_TOKENS,
            temperature=0.30 if attempt == 0 else 0.22,
            top_p=0.76,
            retry=1,
        )
        choices = obj.get("choices", []) if obj else []
        choices = [normalize_choice_surface(c, state) for c in choices]
        ok, probs = validate_choices(choices, state, decision_context)
        if ok:
            return choices, [], raw
        last_probs = probs
        messages = messages + [{"role": "user", "content": "선택지 검증 실패:\n" + "\n".join(probs) + "\n규칙을 지켜 choices JSON만 다시 출력해라."}]
    return choices, last_probs, raw


In [ ]:
# ============================================================
# 12. Drift / summary / state update
# ============================================================

def world_drift_report(state, blueprint, anchor_obj):
    allowed = get_allowed_entities(state, blueprint)
    generated = clean_list(anchor_obj.get("generated_entities", []))
    if not generated:
        generated = anchor_pool(anchor_obj.get("scene_anchors", {}))
    unknown = [e for e in generated if not entity_match(e, allowed)]
    return {
        "allowed_entities": allowed,
        "generated_entities": generated,
        "unknown_entities": unknown,
        "unknown_entity_ratio": len(unknown) / max(1, len(generated))
    }


def summarize_scene(state, story, blueprint, anchors):
    prompt = f"""
TASK: SUMMARIZE_SCENE_STATE

story:
{story}

blueprint:
{json.dumps(blueprint, ensure_ascii=False, indent=2)}

scene_anchors:
{json.dumps(anchors, ensure_ascii=False, indent=2)}

출력 JSON:
{{
  "summary": "한두 문장 요약",
  "new_known_facts": ["..."],
  "new_open_threads": ["..."],
  "new_inventory": ["..."],
  "current_location": "... 또는 null",
  "known_locations": ["..."],
  "known_entities": ["..."]
}}
""".strip()
    obj, raw = generate_json(
        [{"role": "system", "content": SYSTEM_BASE}, {"role": "user", "content": prompt}],
        max_new_tokens=650,
        temperature=0.22,
        top_p=0.72,
        retry=1,
    )
    return obj or {}, raw


def maybe_advance_stage(state):
    order = state["config"]["stage_order"]
    stage = state["current_stage"]
    idx = order.index(stage)
    min_turns = state["config"]["stage_min_turns"].get(stage, 1)
    if state["stage_turns"].get(stage, 0) < min_turns:
        return stage
    if idx + 1 >= len(order):
        return stage
    return order[idx + 1]


def update_state_after_turn(state, story, blueprint, anchors, choices, drift):
    state["last_story"] = story
    sents = split_sentences_ko(story)
    state["last_scene_tail"] = " ".join(sents[-2:])
    summ, _ = summarize_scene(state, story, blueprint, anchors)
    state["last_scene_summary"] = summ.get("summary", state["last_scene_tail"])
    state["known_facts"] = list(dict.fromkeys(state["known_facts"] + clean_list(summ.get("new_known_facts", [])) + clean_list(blueprint.get("new_information", []))))[-12:]
    state["open_threads"] = list(dict.fromkeys(state["open_threads"] + clean_list(summ.get("new_open_threads", [])) + clean_list(blueprint.get("new_problem", []))))[-10:]
    state["inventory"] = list(dict.fromkeys(state["inventory"] + clean_list(summ.get("new_inventory", []))))[-8:]
    loc = summ.get("current_location")
    if loc:
        state["locations"]["current"] = loc
    state["locations"]["known"] = list(dict.fromkeys(state["locations"]["known"] + clean_list(summ.get("known_locations", []))))[-10:]
    state["known_entities"] = list(dict.fromkeys(state["known_entities"] + clean_list(summ.get("known_entities", [])) + anchor_pool(anchors))) [-30:]
    state["last_anchors"] = anchors
    state["last_decision_context"] = {}
    state["last_choices"] = choices
    stage = state["current_stage"]
    state["stage_turns"][stage] = state["stage_turns"].get(stage, 0) + 1
    state["current_stage"] = maybe_advance_stage(state)
    return state


In [ ]:
# ============================================================
# 13. One turn engine
# ============================================================

def generate_turn(state, selected_choice=None):
    if not state.get("story_plan"):
        plan, _ = create_story_plan(state)
        print("[STORY PLAN]")
        pprint(state["story_plan"])

    print("\n[STAGE]")
    print(state.get("current_stage"))

    blueprint, _ = create_scene_blueprint(state, selected_choice=selected_choice)
    if not blueprint:
        print("[BLUEPRINT FAILED]")
        return None

    story, story_probs = write_story_from_blueprint(state, blueprint, selected_choice=selected_choice, retry=2)
    anchor_obj, _ = extract_scene_anchors(story)
    anchors = anchor_obj.get("scene_anchors", {})
    decision_context, _ = make_decision_context(state, story, anchors, blueprint)
    affordances, _ = make_affordances(state, decision_context, anchors)
    choices, choice_probs, raw = make_choices_with_contracts(state, decision_context, affordances)
    drift = world_drift_report(state, blueprint, anchor_obj)

    turn = {
        "stage_before_update": state.get("current_stage"),
        "story": story,
        "story_problems": story_probs,
        "scene_blueprint": blueprint,
        "scene_anchors": anchors,
        "decision_context": decision_context,
        "affordances": affordances,
        "choices": choices,
        "choice_problems": choice_probs,
        "world_drift": drift,
    }

    update_state_after_turn(state, story, blueprint, anchors, choices, drift)
    turn["state_after"] = {
        "current_stage": state["current_stage"],
        "stage_turns": state["stage_turns"],
        "known_facts": state["known_facts"],
        "open_threads": state["open_threads"],
        "locations": state["locations"],
        "inventory": state["inventory"],
    }
    return turn


def print_turn(turn):
    if turn is None:
        return
    print("\n[STORY]")
    print(turn["story"])

    print("\n[STORY_QUALITY]", "ok" if not turn["story_problems"] else "warning")
    for p in turn["story_problems"]:
        print("-", p)

    print("\n[SCENE_BLUEPRINT]")
    pprint(turn["scene_blueprint"])

    print("\n[SCENE_ANCHORS]")
    pprint(turn["scene_anchors"])

    print("\n[DECISION_CONTEXT]")
    pprint(turn["decision_context"])

    print("\n[AFFORDANCES]")
    for a in turn["affordances"]:
        print(f"- {a.get('id')}: {a.get('role')} | anchor={a.get('anchor')} | action={a.get('action')} | risk={a.get('risk')}")

    print("\n[CHOICES]", "ok" if not turn["choice_problems"] else "warning")
    for i, c in enumerate(turn["choices"], 1):
        print(f"{i}. [{c.get('id')}] {c.get('text')}  (anchor={c.get('anchor')}, role={c.get('role')}, risk={c.get('risk')})")
        print("   contract:", c.get("contract"))
    if turn["choice_problems"]:
        print("[CHOICE_PROBLEMS]")
        for p in turn["choice_problems"]:
            print("-", p)

    print("\n[WORLD_DRIFT]")
    pprint(turn["world_drift"])

    print("\n[STATE_AFTER]")
    pprint(turn["state_after"])


In [ ]:
# ============================================================
# 14. Run example
# ============================================================

story_state = init_story_state()
turn1 = generate_turn(story_state)
print_turn(turn1)

selected_idx = 0
if turn1 and turn1.get("choices") and not turn1.get("choice_problems"):
    selected_choice = turn1["choices"][selected_idx]
    print("\n\n[SELECTED]")
    pprint(selected_choice)
    turn2 = generate_turn(story_state, selected_choice=selected_choice)
    print_turn(turn2)
else:
    print("\n선택지 생성이 통과하지 못했습니다. [CHOICE_PROBLEMS]를 확인하세요.")
